# 2. Train (GPU notebook)

- Settings: Accelerator **GPU T4 x2** (preferred; the P100 has no tensor cores), Internet optional.
- Inputs: the cache dataset (or the output of notebook 1), the DieWolke code dataset (unless
  `REPO_URL` is set), and, from the second session on, **this notebook's own previous version**
  (Add Input > Notebook Output), so the runs can resume.
- Run it with **Save Version > Save & Run All**. That runs in the background for up to 12 h with
  no browser open. Training stops on its own ~20 min before the limit, so the checkpoints get
  saved.
- Every session runs the same queue: finished runs are skipped, and unfinished ones continue from
  their last checkpoint.

In [ ]:
import os, glob, shutil, time
T0 = time.time()                      # session start, used to stop training before Kaggle's 12 h limit
REPO_URL = ''                         # optional: git URL of the DieWolke repo, if it is pushed somewhere
CODE = '/kaggle/working/DieWolke'
if not os.path.exists(CODE):
    if REPO_URL:
        !git clone -q {REPO_URL} {CODE}
    else:  # the code uploaded as a Kaggle dataset (the DieWolke folder, without data or runs)
        hits = sorted(glob.glob('/kaggle/input/**/diewolke/__init__.py', recursive=True), key=len)
        assert hits, 'Add the DieWolke code as an input dataset (Add Input > Your Datasets), or set REPO_URL'
        shutil.copytree(os.path.dirname(os.path.dirname(hits[0])), CODE,
                        ignore=shutil.ignore_patterns('*.npy', 'runs', '.git', '.venv', '__pycache__'))
%cd {CODE}
!python kaggle/kaggle_tools.py env

In [ ]:
# runs from earlier versions of this notebook (added as inputs) -> /kaggle/working/runs
!python kaggle/kaggle_tools.py restore
# /kaggle/input is network-backed and far too slow for random training crops: copy the cache to local disk (~2 min)
CACHE = !python kaggle/kaggle_tools.py find-cache --copy_to /kaggle/temp/cache
CACHE = CACHE[-1]; print('cache:', CACHE)

**First session only (optional, ~10 min):** measure speed settings on this GPU. bf16 is not
available on T4/P100, so only fp16 variants are compared. If compile wins, add
`train.compile=true` to `EXTRA` below.

In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    !python scripts/benchmark_speed.py --config configs/full.yaml --set data.cache={CACHE} train.workers=2 train.precision=fp16 \
        --try "train.precision=fp16" --try "train.precision=fp16 train.compile=true" --try "train.precision=fp16 train.channels_last=true"

**The plan (Option A).** With two GPUs the runs alternate between them. Estimates on T4s: about
10-18 GPU hours in total, so roughly 5-9 hours of wall-clock time with both GPUs busy. That's one
or two sessions; check the status table printed at the end. The masker trains first because the
two-stage evaluation uses it.

In [ ]:
EXTRA = ''   # e.g. 'train.compile=true' if the benchmark says so
RUNS = 'masker full@40000 matched_baseline@40000 full:full_20k@20000 no_bga@20000 no_ndvi@20000 no_unc@20000'
!python kaggle/kaggle_tools.py queue {RUNS} --cache {CACHE} --session_start {T0} --set {EXTRA}

In [ ]:
!for d in /kaggle/working/runs/*/; do echo "== $d"; tail -n 2 "$d/log.csv"; done
!du -sh /kaggle/working/runs